# Keras GPU Smoke Test

Notebook simples para validar o ambiente com `keras`, visualizar algumas imagens e treinar uma rede pequena.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from PIL import Image
from tensorflow import keras

In [ ]:
SEED = 42
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data" / "faces"
IMAGES_DIR = DATA_DIR / "final_files"
LABELS_PATH = DATA_DIR / "labels.csv"

print("Projeto:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Labels:", LABELS_PATH.exists(), LABELS_PATH)
print("Imagens:", IMAGES_DIR.exists(), IMAGES_DIR)

In [ ]:
labels = pd.read_csv(LABELS_PATH)
labels["path"] = labels["file_name"].map(lambda name: str(IMAGES_DIR / name))
labels = labels[labels["path"].map(lambda p: Path(p).exists())].copy()
labels["age_group"] = (labels["real_age"] >= 30).astype("int32")

print(labels.head())
print("\nTotal de imagens:", len(labels))
print("Distribuicao da classe age_group:")
print(labels["age_group"].value_counts().sort_index())

In [ ]:
sample_df = labels.sample(9, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, row in zip(axes.flatten(), sample_df.itertuples()):
    image = Image.open(row.path)
    ax.imshow(image)
    ax.set_title(f"idade={row.real_age} | grupo={row.age_group}")
    ax.axis("off")

plt.tight_layout()

In [ ]:
work_df = labels.sample(min(len(labels), 1200), random_state=SEED).reset_index(drop=True)
split_index = int(len(work_df) * 0.8)
train_df = work_df.iloc[:split_index].copy()
val_df = work_df.iloc[split_index:].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_example(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((frame["path"].values, frame["age_group"].values))
    dataset = dataset.map(load_example, num_parallel_calls=AUTOTUNE)
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
    keras.layers.Conv2D(16, 3, activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64, 3, activation="relu"),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2
)

In [ ]:
pd.DataFrame(history.history)